In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from IPython.display import Image
from typing import TypedDict
import os

In [ ]:
model = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="nvidia/nemotron-3.5-lightning:free"
)

In [ ]:
prompt=PromptTemplate(
    template="""Generate single word response for the qiven question:
    {ques}""",
    input_variables=["ques"]
)

In [ ]:
parser=StrOutputParser()

In [ ]:
class BMIState(TypedDict):
    weight_kg:float
    height_m:float
    bmi:float
    bmi_status:str
    status_suggestion:str

In [ ]:
graph=StateGraph(BMIState)

In [ ]:
def calculate_bmi(state:BMIState)->BMIState:
    weight=state['weight_kg']
    height=state['height_m']
    bmi=weight/(height**2)
    state['bmi']=round(bmi, 2)
    return state

In [ ]:
def bmi_status(state:BMIState)->BMIState:
    bmi=state['bmi']
    ques=f"Depending upon the given value of bmi:{bmi} determine wheter the person is underweight, fit or overweight"
    chain=prompt | model | parser
    state['bmi_status']= chain.invoke({'ques':ques})
    return state

In [ ]:
def status_suggestion(state:BMIState)->BMIState:
    bmi_status=state['bmi_status']
    status_suggestion=f"Depending upon the given status of bmi:{bmi_status} among underweight, fit or overweight, determine the type of dite a person can have. Make a diet plan."
    chain=prompt | model | parser
    state['status_suggestion']= chain.invoke({'ques':status_suggestion})
    return state

In [ ]:
graph.add_node('calculate_bmi',calculate_bmi)
graph.add_node('bmi_status',bmi_status)
graph.add_node('status_suggestion',status_suggestion)


In [ ]:
graph.add_edge(START,'calculate_bmi')
graph.add_edge('calculate_bmi','bmi_status')
graph.add_edge('bmi_status','status_suggestion')
graph.add_edge('status_suggestion',END)

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


In [ ]:
# Execute the workflow
final_graph = workflow.invoke({
    "weight_kg": 80,
    "height_m": 1.74
})

print(final_graph)

APIStatusError: Error code: 402 - {'error': 'You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.'}

In [ ]:
display(Image(workflow.get_graph().draw_mermaid_png()))